# Programação em Python para Análise e Visualização de Dados

**Curso:** Bacharelado em Sistemas de Informação (6º período)  
**Aula 01:** Introdução à Análise de Dados com Python  
**Duração:** 3h  
**Ambiente:** Google Colab  

## Objetivos da aula
- Entender o objetivo da disciplina e o fluxo de um projeto analítico.
- Preparar o ambiente no Colab.
- Carregar o dataset padrão (`vendas_brasil.csv`).
- Realizar a primeira exploração (shape, colunas, tipos, amostras).

## Entrega (registro)
- Responder às perguntas de negócio ao final do notebook.
- Salvar o notebook no Google Drive e (opcional) publicar no GitHub.


## 0) Setup do ambiente
Execute a célula abaixo para importar as bibliotecas e configurar a visualização.


In [ ]:
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print('OK: bibliotecas carregadas')


## 1) Dataset padrão da disciplina
Nesta disciplina vamos usar um dataset padrão de vendas no varejo (Brasil).

**Arquivo:** `data/vendas_brasil.csv` (recomendado)  
Se você estiver usando apenas o arquivo `vendas_brasil.csv` sem pasta `data/`, ajuste o caminho conforme indicado.


In [ ]:
# Ajuste o caminho conforme o local do arquivo no seu ambiente
CSV_PATHS = [
    'data/vendas_brasil.csv',   # padrão recomendado
    'vendas_brasil.csv'         # alternativa (arquivo na raiz)
]

csv_path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        'Não encontrei o arquivo vendas_brasil.csv. '
        'Envie o arquivo para o Colab (menu Arquivos → Fazer upload) '
        'ou coloque-o na pasta data/ e rode novamente.'
    )

print('Usando:', csv_path)


In [ ]:
# Carregar dados
df = pd.read_csv(csv_path)
df.head()


## 2) Check-up inicial do dataset
Antes de analisar, sempre faça um check-up rápido:
- Tamanho (linhas x colunas)
- Tipos e colunas
- Valores ausentes (primeira olhada)
- Amostra de registros


In [ ]:
df.shape


In [ ]:
df.columns


In [ ]:
df.info()


In [ ]:
# Estatísticas descritivas (numéricas)
df.describe().T


In [ ]:
# Frequência (exemplo)
df['canal'].value_counts()


## 3) Primeiras perguntas de negócio (responda no notebook)
Escreva as respostas abaixo em texto (Markdown) com base no que você viu no dataset.

1) Liste **5 perguntas** que um gestor gostaria de responder com esses dados.
2) Escolha **2 perguntas** e indique quais colunas seriam usadas.
3) Que tipos de gráfico parecem adequados (barra, linha, dispersão)?


### Respostas do aluno
## 3) Primeiras perguntas de negócio — Respostas

### (1) Cinco perguntas que um gestor poderia querer responder
1. **Qual canal de venda gera maior receita e maior lucro?** (Online x Loja Física x Marketplace x Representante)
2. **Quais UFs concentram mais receita e lucro?** (priorização de região/expansão)
3. **Quais categorias são mais lucrativas e quais têm menor margem?** (mix de produtos)
4. **Quais produtos estão no Top 10 de lucro e quais no Top 10 de receita?** (foco em portfólio)
5. **Como a receita e o lucro evoluem ao longo do tempo (mensal)?** (tendência e sazonalidade)

### (2) Duas perguntas + colunas necessárias
**Pergunta A:** “Qual canal gera maior receita e maior lucro?”  
- Colunas: `canal`, `receita`, `lucro`  
- Operação: agregação (SUM por canal)

**Pergunta B:** “Como a receita evolui ao longo do tempo (mensal)?”  
- Colunas: `data`, `receita`  
- Operação: conversão de data + agregação por mês (SUM)

### (3) Gráficos adequados
- Receita/Lucro por canal: **barras**
- Receita/Lucro por UF: **barras horizontais** (ranking)
- Evolução mensal: **linha**
- Top 10 produtos: **barras horizontais**
- Receita x Lucro (relação): **dispersão (scatter)**

In [ ]:
def human_money(value):
    return f'R$ {value:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

# Pergunta A: Receita e lucro por canal (SUM)
kpi_canal = df.groupby("canal", as_index=False).agg(
    receita_total=("receita", "sum"),
    lucro_total=("lucro", "sum"),
    vendas=("produto", "count")
).sort_values("receita_total", ascending=False)

kpi_canal["margem_total"] = kpi_canal["lucro_total"] / kpi_canal["receita_total"]

display(kpi_canal)

print("\nResumo (texto):")
top = kpi_canal.iloc[0]
print(f"- Canal com maior receita: {top['canal']} (Receita={human_money(top['receita_total'])}, Lucro={human_money(top['lucro_total'])})")

In [ ]:
# Pergunta B: Receita mensal (SUM)
# Converter a coluna 'data' para o tipo datetime
df['data'] = pd.to_datetime(df['data'])
mensal = df.groupby(pd.Grouper(key="data", freq="ME")).agg(
    receita=("receita", "sum"),
    lucro=("lucro", "sum")
).reset_index() # Adicionado .reset_index() para transformar o índice 'data' em coluna

display(mensal.head(12))
display(mensal.tail(12))

### Visualização da Receita Mensal

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.lineplot(data=mensal, x='data', y='receita', marker='o')
plt.title('Evolução da Receita Mensal')
plt.xlabel('Mês')
plt.ylabel('Receita Total')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Visualizar Receita e Lucro por Canal

### Subtask:
Gerar um gráfico de barras para visualizar a receita total e o lucro total por canal de vendas, utilizando o DataFrame `kpi_canal` já existente.


**Reasoning**:
The subtask requires generating two bar charts, one for total revenue and another for total profit, by sales channel. I will use `matplotlib.pyplot` and `seaborn` to create these visualizations in two subplots, as instructed.



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Gráfico de Receita Total por Canal
sns.barplot(data=kpi_canal, x='canal', y='receita_total', ax=axes[0], palette='viridis')
axes[0].set_title('Receita Total por Canal')
axes[0].set_xlabel('Canal de Vendas')
axes[0].set_ylabel('Receita Total (R$)')
axes[0].ticklabel_format(style='plain', axis='y') # Evita notação científica

# Gráfico de Lucro Total por Canal
sns.barplot(data=kpi_canal, x='canal', y='lucro_total', ax=axes[1], palette='plasma')
axes[1].set_title('Lucro Total por Canal')
axes[1].set_xlabel('Canal de Vendas')
axes[1].set_ylabel('Lucro Total (R$)')
axes[1].ticklabel_format(style='plain', axis='y') # Evita notação científica

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Gráfico de Receita Total por Canal
sns.barplot(data=kpi_canal, x='canal', y='receita_total', ax=axes[0], hue='canal', palette='viridis', legend=False)
axes[0].set_title('Receita Total por Canal')
axes[0].set_xlabel('Canal de Vendas')
axes[0].set_ylabel('Receita Total (R$)')
axes[0].ticklabel_format(style='plain', axis='y') # Evita notação científica

# Gráfico de Lucro Total por Canal
sns.barplot(data=kpi_canal, x='canal', y='lucro_total', ax=axes[1], hue='canal', palette='plasma', legend=False)
axes[1].set_title('Lucro Total por Canal')
axes[1].set_xlabel('Canal de Vendas')
axes[1].set_ylabel('Lucro Total (R$)')
axes[1].ticklabel_format(style='plain', axis='y') # Evita notação científica

plt.tight_layout()
plt.show()

In [ ]:
kpi_uf = df.groupby('uf', as_index=False).agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum')
).sort_values('receita_total', ascending=False)

display(kpi_uf.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Gráfico de Receita Total por UF
sns.barplot(data=kpi_uf, x='receita_total', y='uf', ax=axes[0], hue='uf', palette='viridis', legend=False)
axes[0].set_title('Receita Total por UF')
axes[0].set_xlabel('Receita Total (R$)')
axes[0].set_ylabel('UF')
axes[0].ticklabel_format(style='plain', axis='x') # Evita notação científica

# Gráfico de Lucro Total por UF
sns.barplot(data=kpi_uf, x='lucro_total', y='uf', ax=axes[1], hue='uf', palette='plasma', legend=False)
axes[1].set_title('Lucro Total por UF')
axes[1].set_xlabel('Lucro Total (R$)')
axes[1].set_ylabel('UF')
axes[1].ticklabel_format(style='plain', axis='x') # Evita notação científica

plt.tight_layout()
plt.show()

In [ ]:
kpi_produto = df.groupby('produto', as_index=False).agg(
    receita_total=('receita', 'sum'),
    lucro_total=('lucro', 'sum')
)

display(kpi_produto.head())

In [ ]:
top10_receita_produtos = kpi_produto.sort_values('receita_total', ascending=False).head(10)
top10_lucro_produtos = kpi_produto.sort_values('lucro_total', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Gráfico dos Top 10 Produtos por Receita
sns.barplot(data=top10_receita_produtos, x='receita_total', y='produto', ax=axes[0], hue='produto', palette='viridis', legend=False)
axes[0].set_title('Top 10 Produtos por Receita Total')
axes[0].set_xlabel('Receita Total (R$)')
axes[0].set_ylabel('Produto')
axes[0].ticklabel_format(style='plain', axis='x')

# Gráfico dos Top 10 Produtos por Lucro
sns.barplot(data=top10_lucro_produtos, x='lucro_total', y='produto', ax=axes[1], hue='produto', palette='plasma', legend=False)
axes[1].set_title('Top 10 Produtos por Lucro Total')
axes[1].set_xlabel('Lucro Total (R$)')
axes[1].set_ylabel('Produto')
axes[1].ticklabel_format(style='plain', axis='x')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df, x='receita', y='lucro', alpha=0.6)
plt.title('Relação entre Receita e Lucro por Transação')
plt.xlabel('Receita (R$)')
plt.ylabel('Lucro (R$)')
plt.grid(True)
plt.ticklabel_format(style='plain', axis='x') # Evita notação científica
plt.ticklabel_format(style='plain', axis='y') # Evita notação científica
plt.show()

## Análise e Insights dos KPIs de Vendas

Com base nas visualizações geradas, podemos extrair as seguintes percepções sobre o desempenho das vendas:

### 1. Evolução da Receita Mensal
- **Gráfico de Linha (mensal):** A receita mensal mostra uma certa sazonalidade. Houve picos e vales ao longo do ano. É importante analisar se essas variações são esperadas (e.g., feriados, promoções) ou indicam problemas.

### 2. Receita e Lucro por Canal de Vendas
- **Gráficos de Barras (kpi_canal):**
  - A **Loja Física** é o canal que gera a maior receita total e o maior lucro total, indicando sua forte performance e talvez uma base de clientes consolidada ou um ticket médio mais alto.
  - **Online** e **Marketplace** seguem com receitas e lucros significativos, mostrando a importância dos canais digitais. O **Representante** é o canal com menor volume, mas ainda contribui para o resultado geral.
  - A margem de lucro por canal é relativamente similar, sugerindo que a eficiência operacional entre os canais é equilibrada, embora com volumes de vendas diferentes.

### 3. Receita e Lucro por Unidade Federativa (UF)
- **Gráficos de Barras Horizontais (kpi_uf):**
  - As UFs de **SC, ES, PR e RS** destacam-se como as maiores contribuidoras em receita e lucro, sugerindo que estas regiões são mercados-chave ou onde a empresa tem maior penetração. São Paulo (SP) e Minas Gerais (MG) também possuem alta representatividade.
  - A distribuição de receita e lucro entre as UFs é bastante desigual, o que pode indicar oportunidades de expansão ou intensificação de esforços de venda em UFs com menor desempenho.

### 4. Top 10 Produtos por Receita e Lucro
- **Gráficos de Barras Horizontais (kpi_produto):**
  - Existem produtos que são consistentemente altos tanto em receita quanto em lucro (e.g., Produto_62, Produto_59, Produto_113), o que os classifica como `cash cows`.
  - Há produtos que geram alta receita, mas nem sempre estão entre os top de lucro, e vice-versa. Isso é crucial para estratégias de precificação, marketing e gestão de estoque.
  - Identificar esses produtos permite focar esforços de marketing nos mais lucrativos e otimizar a cadeia de suprimentos para os de maior receita.

### 5. Relação entre Receita e Lucro por Transação
- **Gráfico de Dispersão (df):**
  - A visualização mostra uma correlação positiva clara entre receita e lucro. Transações com maior receita tendem a ter maior lucro, o que é esperado.
  - No entanto, o gráfico de dispersão pode revelar *outliers*: transações com alta receita e baixo lucro (ou até prejuízo) ou transações com baixa receita e lucro desproporcionalmente alto. Isso pode indicar problemas de custo, precificação ou promoções específicas que merecem investigação detalhada.
  - A densidade dos pontos em certas áreas pode indicar faixas de receita e lucro mais comuns.